### 1. Dataset Connection

In [1]:

import json
from pathlib import Path
import pandas as pd

# Locate project root
project_root = Path.cwd()

if not (project_root / "config/data_paths.json").exists():
    project_root = project_root.parent

# Load dataset configuration
with open(project_root / "config/data_paths.json") as f:
    config = json.load(f)

dataset_root = Path(config["dataset_root"])

print("Dataset found:", dataset_root.is_dir())
print("Dataset location:", dataset_root)


Dataset found: True
Dataset location: C:\Users\WINSYS\Desktop\Research project\Datasets\MXD\MXD Raw Data


### 2. Dataset Structure Overview

In [2]:

# Find the main task folders (Composite and Wire Harness)
task_folders = sorted(
    folder for folder in dataset_root.iterdir()
    if folder.is_dir()
)

# Initialize the total CSV file counter
total_csvs = 0

# Loop through each task folder
for task in task_folders:

    # Find participant folders (P1, P2, P3, ...)
    participants = sorted(
        folder for folder in task.iterdir()
        if folder.is_dir() and folder.name.startswith("P")
    )

    # Find CSV files inside each participant's Processed Files folder
    csv_files = [
        file
        for participant in participants
        for file in (participant / "Processed Files").rglob("*.csv")
    ]

    # Count the processed CSV files
    total_csvs += len(csv_files)

    # Display results for each task
    print(f"Task: {task.name}")
    print(f"Participants: {len(participants)}")
    print(f"Processed CSV files: {len(csv_files)}")
    print("-" * 40)

# Display the total number of processed CSV files
print(f"Total processed CSV files: {total_csvs}")


Task: Composite
Participants: 43
Processed CSV files: 252
----------------------------------------
Task: Wire Harness
Participants: 43
Processed CSV files: 217
----------------------------------------
Total processed CSV files: 469



### 3. CSV File Organization

Examine the number of CSV files per participant and identify the available experimental sessions.


In [3]:

from collections import Counter

# Examine each task
for task in task_folders:

    # Find participant folders
    participants = sorted(
        folder for folder in task.iterdir()
        if folder.is_dir() and folder.name.startswith("P")
    )

    # Count processed CSV files for each participant
    file_counts = {
        p.name: len(list((p / "Processed Files").glob("*.csv")))
        for p in participants
    }

    # Summarize how many participants have each file count
    count_summary = Counter(file_counts.values())

    print(f"\nTask: {task.name}")
    print("CSV count distribution:", dict(sorted(count_summary.items())))

    # Identify participants with fewer than 7 CSV files
    incomplete = [p for p, count in file_counts.items() if count < 7]
    print("Participants with fewer than 7 files:", incomplete)

    # Display example CSV filenames for P1
    example_folder = task / "P1" / "Processed Files"

    print("P1 CSV files:")
    for file in sorted(example_folder.glob("*.csv")):
        print(" -", file.name)



Task: Composite
CSV count distribution: {0: 7, 7: 36}
Participants with fewer than 7 files: ['P15', 'P23', 'P24', 'P25', 'P26', 'P35', 'P9']
P1 CSV files:
 - Final_Rest_P001C001S002.csv
 - Initial_Rest_P001C001S002.csv
 - Rep1_P001C001S002.csv
 - Rep2_P001C001S002.csv
 - Rep3_P001C001S002.csv
 - Rep4_P001C001S002.csv
 - Rep5_P001C001S002.csv

Task: Wire Harness
CSV count distribution: {0: 12, 7: 31}
Participants with fewer than 7 files: ['P10', 'P13', 'P18', 'P23', 'P24', 'P25', 'P26', 'P3', 'P31', 'P34', 'P35', 'P6']
P1 CSV files:
 - Final_Rest_P001Z001S003.csv
 - Initial_Rest_P001Z001S003.csv
 - Rep1_P001Z001S003.csv
 - Rep2_P001Z001S003.csv
 - Rep3_P001Z001S003.csv
 - Rep4_P001Z001S003.csv
 - Rep5_P001Z001S003.csv


### Inspect CSV file contenets to undersatd the structure

In [10]:

# Select the Composite P1 processed files folder
sample_folder = dataset_root / "Composite" / "P1" / "Processed Files"

# Find the Rep1 CSV file
sample_file = next(sample_folder.glob("Rep1_*.csv"))

# Load only the first 5 rows
sample_df = pd.read_csv(sample_file, nrows=10)

# Display the selected filename
print("File:", sample_file.name)

# Display total columns and their names
print(f"\nTotal columns: {len(sample_df.columns)}")

for column in sample_df.columns:
    print(column)

sample_df.head(10)


File: Rep1_P001C001S002.csv

Total columns: 51
Time
HR_Processed
HR
HRV
RR
RRSQI
ECGSQI
ECG
acc_X
acc_Y
acc_Z
Temperature
IMU_1_ax_g_
IMU_1_ay_g_
IMU_1_az_g_
IMU_1_gx_dps_
IMU_1_gy_dps_
IMU_1_gz_dps_
IMU_2_ax_g_
IMU_2_ay_g_
IMU_2_az_g_
IMU_2_gx_dps_
IMU_2_gy_dps_
IMU_2_gz_dps_
IMU_3_ax_g_
IMU_3_ay_g_
IMU_3_az_g_
IMU_3_gx_dps_
IMU_3_gy_dps_
IMU_3_gz_dps_
IMU_4_ax_g_
IMU_4_ay_g_
IMU_4_az_g_
IMU_4_gx_dps_
IMU_4_gy_dps_
IMU_4_gz_dps_
IMU_5_ax_g_
IMU_5_ay_g_
IMU_5_az_g_
IMU_5_gx_dps_
IMU_5_gy_dps_
IMU_5_gz_dps_
Physical Fatigue - Initial
Physical Fatigue - Final
Mental Fatigue - Initial
Mental Fatigue - Final
Performance rating
Age
Weight
Height
Weights added


,Time,HR_Processed,HR,HRV,RR,RRSQI,ECGSQI,ECG,acc_X,acc_Y,...,IMU_5_gz_dps_,Physical Fatigue - Initial,Physical Fatigue - Final,Mental Fatigue - Initial,Mental Fatigue - Final,Performance rating,Age,Weight,Height,Weights added
0,522500,102.5,130.466306,22.402814,15.9,0.496,0.811,0.823492,0.066127,0.043541,...,14.436704,1,3,1,1,92.91,24,216,190,35
1,523000,104.7,130.434573,22.376775,15.9,0.496,0.811,-0.650494,-1.324813,0.082063,...,1.233281,1,3,1,1,92.91,24,216,190,35
2,523500,108.2,130.476463,22.351200,15.9,0.496,0.804,-3.303531,-0.625702,-0.055543,...,13.288418,1,3,1,1,92.91,24,216,190,35
3,524000,108.2,130.314845,22.326121,15.9,0.496,0.804,1.942350,-0.054765,-0.012957,...,-34.480159,1,3,1,1,92.91,24,216,190,35
4,524500,108.2,129.849866,22.301569,15.9,0.496,0.804,1.410120,-0.182991,-0.003705,...,-5.812640,1,3,1,1,92.91,24,216,190,35
5,525000,108.2,129.851213,22.277574,15.9,0.496,0.804,0.087020,-0.492419,-0.106889,...,-16.733941,1,3,1,1,92.91,24,216,190,35
6,525500,108.2,129.479952,22.254168,15.9,0.496,0.804,-0.158387,-0.508931,0.031095,...,-6.454106,1,3,1,1,92.91,24,216,190,35
7,526000,108.2,129.405652,22.231382,15.9,0.496,0.804,-0.059286,-0.466193,-0.123889,...,3.563215,1,3,1,1,92.91,24,216,190,35
8,526500,108.2,129.765739,22.209245,15.9,0.496,0.804,-0.068347,-0.497865,0.029170,...,-16.625290,1,3,1,1,92.91,24,216,190,35
9,527000,108.2,129.916332,22.187790,15.9,0.496,0.804,0.017942,-0.477887,-0.037307,...,5.428093,1,3,1,1,92.91,24,216,190,35



### 5. Column Consistency

Compare the columns in Composite and Wire Harness to determine whether their processed data can be combined using a common structure.


In [11]:

# Select a sample CSV from Wire Harness P1
wire_folder = dataset_root / "Wire Harness" / "P1" / "Processed Files"
wire_file = next(wire_folder.glob("Rep1_*.csv"))

# Read only column names without loading sensor data
composite_columns = pd.read_csv(sample_file, nrows=0).columns.tolist()
wire_columns = pd.read_csv(wire_file, nrows=0).columns.tolist()

# Compare the columns in both tasks
print("Composite columns:", len(composite_columns))
print("Wire Harness columns:", len(wire_columns))
print("Same columns and order:", composite_columns == wire_columns)

# Identify columns unique to each task
print("\nOnly in Composite:", sorted(set(composite_columns) - set(wire_columns)))
print("Only in Wire Harness:", sorted(set(wire_columns) - set(composite_columns)))


Composite columns: 51
Wire Harness columns: 51
Same columns and order: True

Only in Composite: []
Only in Wire Harness: []


In [12]:

# Find all processed CSV files
processed_files = sorted(
    dataset_root.glob("*/P*/Processed Files/*.csv")
)

# Use the sample CSV columns as the reference
reference_columns = sample_df.columns.tolist()

# Store files with different columns or reading errors
mismatched_files = []
read_errors = []

# Check the column names in every processed CSV
for file in processed_files:
    try:
        columns = pd.read_csv(file, nrows=0).columns.tolist()

        # Identify files with different columns or column order
        if columns != reference_columns:
            mismatched_files.append(str(file.relative_to(dataset_root)))

    except Exception as error:
        read_errors.append((str(file.relative_to(dataset_root)), str(error)))

# Display the verification results
print("Total CSV files:", len(processed_files))
print("Files with different columns:", len(mismatched_files))
print("Files with reading errors:", len(read_errors))

# Display any problematic files
print("\nDifferent columns:", mismatched_files[:10])
print("Reading errors:", read_errors[:10])


Total CSV files: 469
Files with different columns: 0
Files with reading errors: 0

Different columns: []
Reading errors: []
